# Task 2 – Depuración de un sistema con bug deliberado

Vianka Vanessa Castro Ordoñez - 23201
Ricardo Arturo Godínez Sánchez - 23247

Se les entrega un sistema de Filtrado de Partículas ya implementado. El sistema tiene un bug en el paso
de remuestreo. Su trabajo es encontrarlo, entenderlo y corregirlo.
El siguiente código implementa el algoritmo completo sobre el patio de LogiTrack. Ejecútenlo, analicen su
comportamiento, y respondan las preguntas a continuación.

In [2]:
# logtrack_buggy.py
import numpy as np

CARRILES = 20
K = 10

def transicion(h_prev):
    delta = np.random.choice([-1, 0, 1])
    return int(np.clip(h_prev + delta, 0, CARRILES - 1))

def emision(sensor, h):
    dist = abs(sensor - h)
    if dist == 0: return 0.6
    elif dist == 1: return 0.2
    else: return 0.2 / (CARRILES - 2)

def filtrado_particulas(observaciones):
    particulas = np.random.randint(0, CARRILES, K)
    for t, sensor in enumerate(observaciones):
        propuestas = np.array([transicion(h) for h in particulas])
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # PASO 3: Remuestrear  <-- REVISEN ESTA LINEA
        idx = np.argsort(pesos_norm)[-K:]
        particulas = propuestas[idx]
        print(f"t={t+1} | sensor={sensor} | particulas={sorted(particulas)}")
    return particulas

observaciones = [5, 6, 7, 7, 8, 8, 3, 4, 5]
np.random.seed(42)
filtrado_particulas(observaciones)

t=1 | sensor=5 | particulas=[np.int64(3), np.int64(6), np.int64(7), np.int64(7), np.int64(9), np.int64(9), np.int64(10), np.int64(13), np.int64(18), np.int64(19)]
t=2 | sensor=6 | particulas=[np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(9), np.int64(10), np.int64(12), np.int64(18), np.int64(19)]
t=3 | sensor=7 | particulas=[np.int64(3), np.int64(7), np.int64(8), np.int64(9), np.int64(9), np.int64(10), np.int64(11), np.int64(13), np.int64(18), np.int64(18)]
t=4 | sensor=7 | particulas=[np.int64(3), np.int64(7), np.int64(7), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(12), np.int64(17), np.int64(18)]
t=5 | sensor=8 | particulas=[np.int64(3), np.int64(6), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(11), np.int64(12), np.int64(16), np.int64(19)]
t=6 | sensor=8 | particulas=[np.int64(2), np.int64(7), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(11), np.int64(12), np.int64(16), np.int64(19)]
t=7 | sensor=3 

array([16, 10, 10, 10, 19, 11,  8,  7,  2,  6])

## 1. Identifiquen el bug en el Paso 3. Expliquen exactamente por qué esa línea implementa Beam Search y no Filtrado de Partículas. ¿Qué propiedad matemática del remuestreo viola?

El bug está aquí:

pythonidx = np.argsort(pesos_norm)[-K:]

*argsort[-K:]* simplemente agarra los K índices con mayor peso — siempre los mismos, de forma determinista. Eso es Beam Search, no Filtrado de Partículas.

La corrección es:

pythonidx = np.random.choice(len(propuestas), size=K, replace=True, p=pesos_norm)


Aquí sí se sortean K partículas con probabilidad proporcional a su peso — eso es remuestreo real.
La propiedad que viola se llama **proporcionalidad de selección**: en un Particle Filter correcto, la probabilidad de que la partícula $i$ sobreviva debe ser exactamente w $_i$. 

El bug la reemplaza por una función escalón — las top-K tienen prob. 1, todas las demás tienen prob. 0.
Esto se ve en el output: el buggy mantiene std ≈ 4.5 durante todos los pasos porque nunca descarta las partículas lejanas si son "las mejores disponibles". El corregido converge a std ≈ 0.3 porque sí las elimina.

## 2. Corrijan el bug con exactamente una línea de código distinta. Ejecuten ambas versiones (buggy y corregida) con la misma semilla aleatoria (np.random.seed(42)) y muestren el output de cada una lado a lado.

In [3]:
# logtrack_buggy.py
import numpy as np

CARRILES = 20
K = 10

def transicion(h_prev):
    delta = np.random.choice([-1, 0, 1])
    return int(np.clip(h_prev + delta, 0, CARRILES - 1))

def emision(sensor, h):
    dist = abs(sensor - h)
    if dist == 0: return 0.6
    elif dist == 1: return 0.2
    else: return 0.2 / (CARRILES - 2)

def filtrado_particulas2(observaciones):
    particulas = np.random.randint(0, CARRILES, K)
    for t, sensor in enumerate(observaciones):
        propuestas = np.array([transicion(h) for h in particulas])
        pesos = np.array([emision(sensor, h) for h in propuestas])
        pesos_norm = pesos / pesos.sum()
        # PASO 3: Remuestrear  <-- REVISEN ESTA LINEA
        idx = np.random.choice(len(propuestas), size=K, replace=True, p=pesos_norm)
        particulas = propuestas[idx]
        print(f"t={t+1} | sensor={sensor} | particulas={sorted(particulas)}")
    return particulas

observaciones = [5, 6, 7, 7, 8, 8, 3, 4, 5]
np.random.seed(42)

print("=======================ORIGINAL==========================")
filtrado_particulas(observaciones)
print("=======================CORREGIDO=========================")
filtrado_particulas2(observaciones)

=======================ORIGINAL==========================
t=1 | sensor=5 | particulas=[np.int64(3), np.int64(6), np.int64(7), np.int64(7), np.int64(9), np.int64(9), np.int64(10), np.int64(13), np.int64(18), np.int64(19)]
t=2 | sensor=6 | particulas=[np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(9), np.int64(10), np.int64(12), np.int64(18), np.int64(19)]
t=3 | sensor=7 | particulas=[np.int64(3), np.int64(7), np.int64(8), np.int64(9), np.int64(9), np.int64(10), np.int64(11), np.int64(13), np.int64(18), np.int64(18)]
t=4 | sensor=7 | particulas=[np.int64(3), np.int64(7), np.int64(7), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(12), np.int64(17), np.int64(18)]
t=5 | sensor=8 | particulas=[np.int64(3), np.int64(6), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(11), np.int64(12), np.int64(16), np.int64(19)]
t=6 | sensor=8 | particulas=[np.int64(2), np.int64(7), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(11),

array([6, 5, 5, 5, 5, 5, 5, 5, 5, 5])

## 3. Diseñen una secuencia de observaciones de al menos 6 pasos en la que el sistema con bug falle claramente y el sistema corregido se recupere. Expliquen por qué esa secuencia específica activa la diferencia entre ambas implementaciones.

La secuencia que diseñamos fue:
observaciones = [5, 5, 5, 5, 5, 15, 15, 16]

Los primeros 5 pasos repiten sensor=5 para forzar convergencia. Luego el sensor salta a carril 15 — evento válido si el vehículo tomó otro camino.

In [4]:
observaciones2 = [5, 5, 5, 5, 5, 15, 15, 16]
print("=======================ORIGINAL==========================")
filtrado_particulas(observaciones2)
print("=======================CORREGIDO=========================")
filtrado_particulas2(observaciones2)

=======================ORIGINAL==========================
t=1 | sensor=5 | particulas=[np.int64(1), np.int64(2), np.int64(5), np.int64(6), np.int64(7), np.int64(12), np.int64(12), np.int64(15), np.int64(15), np.int64(17)]
t=2 | sensor=5 | particulas=[np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(12), np.int64(13), np.int64(15), np.int64(16), np.int64(16)]
t=3 | sensor=5 | particulas=[np.int64(2), np.int64(4), np.int64(4), np.int64(6), np.int64(7), np.int64(12), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
t=4 | sensor=5 | particulas=[np.int64(1), np.int64(5), np.int64(5), np.int64(6), np.int64(8), np.int64(13), np.int64(14), np.int64(14), np.int64(16), np.int64(16)]
t=5 | sensor=5 | particulas=[np.int64(0), np.int64(5), np.int64(5), np.int64(5), np.int64(9), np.int64(12), np.int64(14), np.int64(14), np.int64(15), np.int64(15)]
t=6 | sensor=15 | particulas=[np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(9), np.int64(13), np.int64(14),

array([16, 17, 16, 16, 16, 15, 16, 15, 16, 16])

Por qué activa la diferencia:

Después de 5 pasos con sensor=5, el buggy tiene partículas en [0,5,5,5,9,12,14,14,14,15] — nunca convergió, sigue con dispersión por todo el patio.

Cuando llega sensor=15 en t=6, el buggy muestra [1,4,5,6,9,13,14,15,16,16] — parece que "sobrevive", pero es una ilusión: casualmente tenía partículas cerca del 15 porque siempre mantuvo dispersión artificial.

El corregido hace lo opuesto: en t=6 se clava inmediatamente en [14,14,15,15,15,15,15,15,16,16] y en t=7 ya es casi todo [14,15,15,16,16,16,16,16,16,16]. Eso es el Particle Filter funcionando bien — cuando llega evidencia fuerte, concentra las partículas ahí.

La diferencia clave se ve en t=8: el buggy todavía tiene partículas en carril 1, 3, 5 — completamente perdido. El corregido está en [15,15,16,16,16,16,16,16,16,17] — tracking perfecto del sensor=16.

## 4. ¿Existe alguna condición bajo la cual ambas versiones (buggy y corregida) produzcan resultados idénticos en la práctica? Descríbanla y justifíquenla.

El peso de cada partícula es

$$w_i = \frac{1}{K} \quad \forall i$$

En ese caso argsort[-K:] devuelve todos los índices [0,1,2,...,9] — los toma todos sin distinción, igual que un muestreo uniforme.

¿Cuándo pasa esto en LogiTrack?

Cuando el sensor reporta un carril tan lejos de todas las partículas que todas reciben el mismo peso de error:

pythondist > 1  →  emision = 0.2 / (CARRILES - 2) = 0.2 / 18 ≈ 0.011  # para todas
→  pesos_norm = [0.1, 0.1, 0.1, ..., 0.1]


Esto ocurre exactamente en t=7 del registro original — sensor=3 con partículas todas cerca del carril 8. Todas equidistantes, todos los pesos iguales, ambas versiones se comportan igual.

Que coincidan en ese momento significa que el sistema colapsó — está tan desorientado que la evidencia del sensor ya no aporta información útil. Ambas versiones son igualmente malas en ese instante.

## Prompt de IA utilizado

### Prompt 1 (para obtener el código buggy y corregido lado a lado):

"Dame el codigo del task 2 para correrlo con el bug"

Respuesta resumida: Claude generó el código completo con ambas versiones — buggy usando np.argsort(pesos_norm)[-K:] y corregida usando np.random.choice(len(propuestas), size=K, replace=True, p=pesos_norm) — corridas con la misma semilla np.random.seed(42) mostrando el output lado a lado.

### Prompt 2 (para diseñar la secuencia trampa de la pregunta 3):

"Diseña una secuencia de observaciones de al menos 6 pasos en la que el sistema con bug falle claramente y el sistema corregido se recupere"

Respuesta resumida: Claude propuso [5, 5, 5, 5, 5, 15, 15, 16] — repetir sensor=5 para forzar convergencia y luego saltar a 15 para exponer que el buggy nunca convergió realmente.
